# RubikVision · YOLO26s (Colab)

**6 màu:** `b, g, o, r, w, y` · **Dataset:** `rubik_color.zip` · **GPU:** Colab T4.

Chạy tuần tự từ trên xuống. **Chỉ sửa cell Cấu hình**. `MODE = "eval"` chỉ đánh giá model sẵn có; `"train"` bắt đầu run mới; `"resume"` tiếp tục run *bị gián đoạn*. Run đã kết thúc bằng Early Stopping thì dùng `eval` hoặc tạo run mới, không resume.


## 1 · Cài đặt

In [1]:
%pip -q install "ultralytics==8.4.216" "pyyaml>=6.0,<7"

## 2 · Cấu hình & Google Drive

In [2]:
from pathlib import Path
import os, shutil, zipfile, json
import yaml
import torch
from google.colab import drive
from ultralytics import YOLO

drive.mount('/content/drive')
assert torch.cuda.is_available(), "Colab > Runtime > Change runtime type > T4 GPU"
print('GPU:', torch.cuda.get_device_name(0))

# === CHỈ SỬA Ở ĐÂY ===
MODE = 'train'  # 'train' | 'resume' | 'eval'
RUN_NAME = 'yolo26s_rubik_v2'  # Đổi thành v2 nếu train mới
ZIP_NAME = 'rubik_color.zip'
EPOCHS, PATIENCE, BATCH = 150, 30, 32
# =====================

DRIVE = Path('/content/drive/MyDrive/RubikVision')
ZIP = DRIVE / 'datasets' / ZIP_NAME
LOCAL = Path('/content/rubikvision_workspace')
DATA = LOCAL / 'dataset'
DATA_YAML = DATA / 'data.yaml'
RUN_DIR = LOCAL / 'runs' / RUN_NAME
DRIVE_RUN = DRIVE / 'runs' / RUN_NAME
DRIVE_WEIGHTS = DRIVE_RUN / 'weights'
EXPORT_DIR = DRIVE / 'exports' / RUN_NAME
assert MODE in ('train', 'resume', 'eval')
assert ZIP.is_file(), f'Không tìm thấy dataset: {ZIP}'
if MODE == 'train' and (DRIVE_WEIGHTS / 'last.pt').exists():
    raise FileExistsError('Run đã tồn tại trên Drive. Hãy đổi RUN_NAME để tránh ghi đè.')
print('Mode:', MODE, '| Run:', RUN_NAME)


Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
Mounted at /content/drive
GPU: Tesla T4
Mode: train | Run: yolo26s_rubik_v2


## 3 · Chuẩn bị dataset (trên SSD Colab)

In [3]:
if not (DATA / 'train' / 'images').is_dir():
    DATA.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(ZIP) as z:
        for info in z.infolist():
            p = Path(info.filename)
            if p.is_absolute() or '..' in p.parts:
                raise ValueError(f'Unsafe ZIP path: {info.filename}')
        z.extractall(DATA)

cfg = yaml.safe_load((DATA / 'data.yaml').read_text(encoding='utf-8'))
assert cfg['names'] == ['blue', 'green', 'orange', 'red', 'white', 'yellow'], cfg['names']
cfg.update(path=str(DATA), train='train/images', val='valid/images', test='test/images')
DATA_YAML.write_text(yaml.safe_dump(cfg, sort_keys=False), encoding='utf-8')

for split in ('train', 'valid', 'test'):
    images = [p for p in (DATA / split / 'images').iterdir()
              if p.suffix.lower() in {'.jpg','.jpeg','.png','.webp'}]
    labels = list((DATA / split / 'labels').glob('*.txt'))
    assert images and labels, f'Dataset thiếu {split}'
    print(f'{split:5s}: {len(images):4d} ảnh / {len(labels):4d} nhãn')
print('✅ Dataset ready:', DATA_YAML)


train: 2375 ảnh / 2375 nhãn
valid:  339 ảnh /  339 nhãn
test :  181 ảnh /  181 nhãn
✅ Dataset ready: /content/rubikvision_workspace/dataset/data.yaml


## 4 · Checkpoint: lưu lên Drive sau mỗi epoch

In [4]:
LAST = DRIVE_WEIGHTS / 'last.pt'
PREV = DRIVE_WEIGHTS / 'last_prev.pt'
BEST = DRIVE_WEIGHTS / 'best.pt'

def copy_safe(src, dst, backup=None):
    src, dst = Path(src), Path(dst)
    if not src.is_file() or not src.stat().st_size:
        return False
    dst.parent.mkdir(parents=True, exist_ok=True)
    tmp = dst.with_name('.' + dst.name + '.tmp')
    try:
        shutil.copy2(src, tmp)
        if tmp.stat().st_size != src.stat().st_size:
            raise OSError('Copy incomplete')
        if backup is not None and dst.exists():
            old = Path(backup)
            old_tmp = old.with_name('.' + old.name + '.tmp')
            try:
                shutil.copy2(dst, old_tmp)
                if old_tmp.stat().st_size != dst.stat().st_size:
                    raise OSError('Backup incomplete')
                os.replace(old_tmp, old)
            finally:
                old_tmp.unlink(missing_ok=True)
        os.replace(tmp, dst)
        return True
    finally:
        tmp.unlink(missing_ok=True)

def copy_artifacts(src_dir, dst_dir):
    for pattern in ('results.csv', 'args.yaml', '*.png', 'val_batch*_pred.jpg'):
        for f in Path(src_dir).glob(pattern):
            if f.is_file():
                copy_safe(f, Path(dst_dir) / f.name)

def on_save(trainer):
    try:
        if not copy_safe(trainer.last, LAST, PREV):
            return
        if Path(trainer.best).is_file() and (
            not BEST.exists() or trainer.best_fitness == trainer.fitness
        ):
            copy_safe(trainer.best, BEST)
        for name in ('results.csv', 'args.yaml'):
            copy_safe(Path(trainer.save_dir)/name, DRIVE_RUN/name)
        print(f'✅ Synced epoch {trainer.epoch + 1}')
    except Exception as e:
        print('⚠️ Drive sync failed:', e)

def on_finish(trainer):
    try:
        copy_artifacts(trainer.save_dir, DRIVE_RUN)
        # Chỉ xuất model cho inference, không ghi đè last.pt đã sao lưu.
        copy_safe(trainer.best, EXPORT_DIR / 'best_final.pt')
        print('✅ Weights / charts exported')
    except Exception as e:
        print('⚠️ Final export failed:', e)


## 5 · Train / resume (bỏ qua khi `MODE="eval"`)

In [5]:
TRAIN_CONFIG = dict(
    data=str(DATA_YAML), epochs=EPOCHS, patience=PATIENCE,
    imgsz=640, batch=BATCH, device=0, workers=2,
    optimizer='auto', lrf=0.01, box=7.5, cls=0.5, dfl=1.5,
    # Hình học: khối lúc gần lúc xa, hơi nghiêng
    degrees=15, translate=0.10, scale=0.5, shear=3,
    fliplr=0.5, flipud=0.0, perspective=0.0005,
    # Màu: KHÔNG đổi sắc độ (hsv_h=0), vì sẽ biến cam thành đỏ/vàng
    hsv_h=0.0, hsv_s=0.4, hsv_v=0.4, bgr=0.0,
    # Mosaic cho nhiều kích thước và nền; tắt ở 15 epoch cuối cho ổn định
    mosaic=1.0, close_mosaic=15, mixup=0.0,
    seed=42, deterministic=True, amp=True,
    save=True, plots=True,
    project=str(LOCAL / 'runs'), name=RUN_NAME, exist_ok=True,
)

if MODE in ('train', 'resume'):
    if MODE == 'resume':
        resume_file = None
        for p in (LAST, PREV):
            if p.is_file():
                try:
                    state = YOLO(str(p)).ckpt
                    if isinstance(state, dict) and state.get('optimizer') is not None and int(state.get('epoch', -1)) >= 0:
                        resume_file = p
                        break
                except Exception as e:
                    print('Skip bad checkpoint:', p.name, e)
        if resume_file is None:
            raise FileNotFoundError('Không có checkpoint resumable trên Drive.')
        (RUN_DIR / 'weights').mkdir(parents=True, exist_ok=True)
        shutil.copy2(resume_file, RUN_DIR / 'weights' / 'last.pt')
        for f in ('results.csv', 'args.yaml'):
            if (DRIVE_RUN / f).is_file():
                shutil.copy2(DRIVE_RUN / f, RUN_DIR / f)
        if BEST.is_file():
            shutil.copy2(BEST, RUN_DIR / 'weights' / 'best.pt')
        model = YOLO(str(RUN_DIR / 'weights' / 'last.pt'))
    else:
        model = YOLO('yolo26s.pt')

    model.add_callback('on_model_save', on_save)
    model.add_callback('on_train_end', on_finish)
    if MODE == 'resume':
        model.train(resume=True)  # Giữ cấu hình đã lưu trong checkpoint
    else:
        model.train(**TRAIN_CONFIG)
else:
    print('ℹ️ Evaluation-only: không train lại.')

Ultralytics 8.4.216 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=15, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/rubikvision_workspace/dataset/data.yaml, degrees=15, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=150, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.0, hsv_s=0.4, hsv_v=0.4, imgsz=640, iou=None, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolo26s_rubik_v2, nbs=64, nms=None, opset=

## 6 · Đánh giá Validation / Test và lưu biểu đồ

In [6]:
weights = EXPORT_DIR / 'best_final.pt'
if not weights.is_file():
    weights = BEST
assert weights.is_file(), f'Không tìm thấy best.pt tại {weights}'

model = YOLO(str(weights))
report = {}
for split in ('val', 'test'):
    metrics = model.val(
        data=str(DATA_YAML), split=split, imgsz=640, device=0,
        plots=True, project=str(LOCAL / 'evaluation' / RUN_NAME),
        name=split, exist_ok=True,
    )
    report[split] = dict(mAP50=float(metrics.box.map50),
                         mAP50_95=float(metrics.box.map))
    out = DRIVE_RUN / 'evaluation' / split
    copy_artifacts(Path(metrics.save_dir), out)
    out.mkdir(parents=True, exist_ok=True)
    (out / 'metrics.json').write_text(json.dumps(report[split], indent=2))
    print(f"{split.upper()}: mAP50={report[split]['mAP50']:.4f}, "
          f"mAP50-95={report[split]['mAP50_95']:.4f}")

print('✅ Weights:', weights)
print('✅ Training charts:', DRIVE_RUN)


Ultralytics 8.4.216 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
YOLO26s summary (fused): 120 layers, 9,467,502 parameters, 0 gradients, 20.8 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1598.6±598.7 MB/s, size: 36.4 KB)
val: Scanning /content/rubikvision_workspace/dataset/valid/labels.cache... 339 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 339/339 109.4Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 22/22 2.9it/s 7.5s
                   all        339       4949       0.97      0.965      0.978      0.754
                  blue        311        852       0.98      0.985      0.986      0.747
                 green        310        801      0.967      0.972      0.978      0.748
                orange        311        810      0.969      0.965      0.975      0.761
                   red        308        807      0.957      0.943      0.961      0.729
                 whit

## 7 · Kết quả cho README

- **Model:** `exports/<RUN_NAME>/best_final.pt` (dùng inference).
- **Resume:** `runs/<RUN_NAME>/weights/last.pt` (+ `last_prev.pt` backup).
- **Đồ thị:** `runs/<RUN_NAME>/results.png`, `confusion_matrix.png`, `BoxPR_curve.png`, `BoxF1_curve.png`, và `evaluation/{val,test}/`.
- **Lưu ý:** mAP của dataset **không thay thế** kiểm thử Rubik 3×3 bằng webcam thật. Chỉ đưa biểu đồ đã có số liệu thực vào README.
